# Phase 21 — Deep temporal learning I: LSTM & GRU (stage 36)

| | |
| --- | --- |
| **Workflow** | `final workflow.md` **Phase 21** → checklist **stage 36** |
| **Fit by** | `scripts/train_sequence.py` → `scripts/phase21_run.log` (resume-safe; every family `exit=0`) |
| **Reads** | `reports/phase21_metrics.csv` (360 rows) · `reports/phase21_search.csv` (46 configs) · `models/phase21_L3-0{1..5}.pt` · `reports/phase17_experiment_registry.csv` · `reports/phase16_baseline_metrics.csv` · `reports/phase20_metrics.csv` · `data/processed/{sequence_index,sequences_L*,sequence_mask_L*}.npy` |
| **Contract** | Dataset B raw sequences + mask (P19-M1) · 13 channels · per-fold standardisation · dev cells only (test frozen to Phase 33) · budgets ≤ 20 · seed 20261008 |

Every claim in `reports/phase21_temporal_dl.md` is re-derived here from the artefacts.

In [1]:
import hashlib, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")

ROOT = Path(r"D:\Flood predictor")
import sys
SCRIPTS = str(ROOT / "scripts")
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
MODELS = ROOT / "models"
MET = REPORTS / "phase21_metrics.csv"
SEARCH = REPORTS / "phase21_search.csv"
P16 = REPORTS / "phase16_baseline_metrics.csv"
P20 = REPORTS / "phase20_metrics.csv"
REG = REPORTS / "phase17_experiment_registry.csv"
REPORT = REPORTS / "phase21_temporal_dl.md"
LOG = ROOT / "scripts" / "phase21_run.log"
TBL = PROC / "experimental_dataset.csv"

TBL_SHA = "a6748529083fcdceb8bb8a657b067bf12b5c861f3a7a6b09c8977ea63be73c8e"
SEED = 20261008
FAMILIES = ["lstm_L3", "lstm_L7", "lstm_L14", "lstm_L30", "gru_bestL"]
REG_ID = {"lstm_L3": "L3-01", "lstm_L7": "L3-02", "lstm_L14": "L3-03",
          "lstm_L30": "L3-04", "gru_bestL": "L3-05"}
BUDGET = {"lstm_L3": 12, "lstm_L7": 10, "lstm_L14": 8, "lstm_L30": 6,
          "gru_bestL": 10}
FAM_L = {"lstm_L3": 3, "lstm_L7": 7, "lstm_L14": 14, "lstm_L30": 30,
         "gru_bestL": 3}
HORIZONS = ["label_24h", "label_48h", "label_72h"]
DEV_CELLS = {"val_2022", "val_2020", "val_2021", "spatial_fold_1",
             "spatial_fold_2", "spatial_fold_3", "spatial_fold_4"}


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()


RES = []
def check(cid, detail, ok=True):
    RES.append({"id": cid, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {cid}: {detail}")
    assert ok, cid


met = pd.read_csv(MET)
search = pd.read_csv(SEARCH)
reg = pd.read_csv(REG)
p16 = pd.read_csv(P16)
p20 = pd.read_csv(P20)
t0 = time.time()
print("Phase 21 notebook - LSTM/GRU verification")

Phase 21 notebook - LSTM/GRU verification


In [2]:
# N21-01  artefacts present with the expected shapes
for p in (MET, SEARCH, REPORT, LOG):
    assert p.exists(), p
pts = {f: MODELS / f"phase21_{REG_ID[f]}.pt" for f in FAMILIES}
missing = [p.name for p in pts.values() if not p.exists()]
check("N21-01", f"metrics {len(met)} rows, search {len(search)} rows, "
                f"5 model files ({sum(p.stat().st_size for p in pts.values())/1e6:.2f} MB), "
                f"report + run log present; missing={missing}",
      not missing and len(met) == 360 and len(search) == 46)

[PASS] N21-01: metrics 360 rows, search 46 rows, 5 model files (1.70 MB), report + run log present; missing=[]


In [3]:
# N21-02  registry contract (L3-01..L3-05: frozen test, budget <= 20)
l3 = reg[reg.level == "L3"].set_index("model")
fam_rows = l3.loc[FAMILIES]
bad = fam_rows[(fam_rows.test_policy != "frozen_read_at_phase33")
               | (fam_rows.tuning_budget_configs > 20)
               | (~fam_rows.metrics_out.str.contains("phase21_metrics.csv", regex=False))
               | (fam_rows.selection_cell != "val_2022")
               | (fam_rows.selection_metric != "pr_auc")]
check("N21-02", f"registry: {len(fam_rows)} L3 rows {sorted(FAMILIES)}, "
                f"test_policy all frozen_read_at_phase33, budgets "
                f"{sorted(fam_rows.tuning_budget_configs.unique())} <= 20, "
                f"metrics_out/selection matches; violations={len(bad)}",
      bad.empty and sorted(fam_rows.registry_id) == sorted(REG_ID.values())
      and set(fam_rows.index) == set(FAMILIES))

[PASS] N21-02: registry: 5 L3 rows ['gru_bestL', 'lstm_L14', 'lstm_L3', 'lstm_L30', 'lstm_L7'], test_policy all frozen_read_at_phase33, budgets [np.int64(20)] <= 20, metrics_out/selection matches; violations=0


In [4]:
# N21-03  test policy: no test rows anywhere in the phase-21 metrics
cells = set(met.cell.unique())
test_rows = met[met.cell.astype(str).str.startswith("test")]
seasons = set(met.season.unique())
check("N21-03", f"cells = {sorted(cells)} == the 7 dev cells (no test), "
                f"test rows = {len(test_rows)} (must be 0), seasons = {sorted(seasons)}",
      cells == DEV_CELLS and len(test_rows) == 0
      and seasons == {"all", "monsoon", "non-monsoon"})

[PASS] N21-03: cells = ['spatial_fold_1', 'spatial_fold_2', 'spatial_fold_3', 'spatial_fold_4', 'val_2020', 'val_2021', 'val_2022'] == the 7 dev cells (no test), test rows = 0 (must be 0), seasons = ['all', 'monsoon', 'non-monsoon']


In [5]:
# N21-04  grid identity vs split_assignments + holdout/test exclusion
# The notebook's sequence index carries the same split/fold/holdout columns as
# Phase 15, so the masks are rebuilt here independently of train_sequence.py.
import baseline_model  # noqa: F401  (ensures scripts/ import path works)
from train_sequence import build_tasks_index

ix = pd.read_csv(PROC / "sequence_index.csv", parse_dates=["date"])
assert len(ix) == 200665
hold = ix.station_holdout.astype(bool).to_numpy()
split = ix.split.to_numpy()
year = ix.date.dt.year.to_numpy()
fold = ix.spatial_fold.to_numpy()

rebuilt = {"train_<2022": (split == "train") & ~hold}
dev = {"val_2022": (split == "val") & ~hold}
for Y in (2020, 2021, 2022):
    rebuilt[f"train_<={Y-1}"] = (year <= Y - 1) & ~hold
    dev[f"val_{Y}"] = (year == Y) & ~hold
for k in (1, 2, 3, 4):
    rebuilt[f"excl_fold_{k}"] = (split == "train") & ~hold & (fold != k)
    dev[f"spatial_fold_{k}"] = (split == "train") & (fold == k)

tasks = build_tasks_index(ix)
mism = 0
for t in tasks:
    mism += int(not np.array_equal(t["train"], rebuilt[t["name"]]))
    for _, cell, m in t["evals"]:
        mism += int(not np.array_equal(m, dev[cell]))
leak = {t["name"]: int((t["train"] & hold).sum()
                       + (t["train"] & (split == "test")).sum()) for t in tasks}
check("N21-04", f"8 sequence-index training masks == masks rebuilt from "
                f"split_assignments (mismatches={mism}); dev eval masks 1:1; "
                f"training masks contain {sum(leak.values())} holdout/test rows "
                f"(must be 0): {leak}",
      mism == 0 and sum(leak.values()) == 0)

[PASS] N21-04: 8 sequence-index training masks == masks rebuilt from split_assignments (mismatches=0); dev eval masks 1:1; training masks contain 0 holdout/test rows (must be 0): {'train_<2022': 0, 'train_<=2019': 0, 'train_<=2020': 0, 'train_<=2021': 0, 'excl_fold_1': 0, 'excl_fold_2': 0, 'excl_fold_3': 0, 'excl_fold_4': 0}


In [6]:
# N21-05  stage-36 tuning: budget honoured, 3 horizons, mean == mean, winner == argmax
rows = []
for f in FAMILIES:
    g = search[search.family == f]
    n_cfg = len(g)
    three = bool(((g[["pr_auc_24h", "pr_auc_48h", "pr_auc_72h"]].notna()).all(axis=1)).all())
    mean_ok = bool(np.allclose(g.mean_pr_auc,
                               g[["pr_auc_24h", "pr_auc_48h", "pr_auc_72h"]].mean(axis=1)))
    winner = g.loc[g.mean_pr_auc.idxmax()]
    rows.append((f, n_cfg, BUDGET[f], three, mean_ok, winner.config_id,
                 round(float(winner.mean_pr_auc), 5), winner.weighting,
                 int(winner.n_params)))
tab = pd.DataFrame(rows, columns=["family", "configs", "budget", "3_horizons",
                                  "mean_ok", "winner", "mean_pr_auc",
                                  "weighting", "n_params"])
print(tab.to_string(index=False))
check("N21-05", f"all 5 families: configs == budget, 3 horizons scored, "
                f"mean == mean(3), winner = argmax; budgets all <= 20 = "
                f"{sorted(tab.budget)}; n_params {sorted(tab.n_params)}",
      bool((tab.configs == tab.budget).all() and tab["3_horizons"].all()
           and tab.mean_ok.all() and (tab.budget <= 20).all()))

   family  configs  budget  3_horizons  mean_ok        winner  mean_pr_auc        weighting  n_params
  lstm_L3       12      12        True     True   lstm_L3_c11      0.92677 pos_weight=ratio     14497
  lstm_L7       10      10        True     True   lstm_L7_c07      0.91146 pos_weight=ratio     53569
 lstm_L14        8       8        True     True  lstm_L14_c04      0.90775             none     14497
 lstm_L30        6       6        True     True  lstm_L30_c05      0.91688             none     14497
gru_bestL       10      10        True     True gru_bestL_c07      0.92362 pos_weight=ratio     40193
[PASS] N21-05: all 5 families: configs == budget, 3 horizons scored, mean == mean(3), winner = argmax; budgets all <= 20 = [6, 8, 10, 10, 12]; n_params [14497, 14497, 14497, 40193, 53569]


In [7]:
# N21-06  stage-36 grid: full 8-cell x 3-horizon x 3-season grid, no empty cells
size = met.groupby("model").size()
covered = met.groupby(["model", "protocol", "train", "horizon", "season"]).size()
ci_missing = met[(met.season == "all") & (met.n >= 1000) & met.pr_auc_ci_lo.isna()]
check("N21-06", f"rows per family = {dict(size)} (expect 72 each); "
                f"{len(covered)} family x task x horizon x season cells populated "
                f"(min n_group={int(covered.min())}); n all > 0 = "
                f"{bool((met.n > 0).all())}; bootstrap CI missing on big cells = "
                f"{len(ci_missing)}; n_params present = "
                f"{bool(met.n_params.notna().all())}",
      (size == 72).all() and len(covered) == 360 and (met.n > 0).all()
      and len(ci_missing) == 0 and bool(met.n_params.notna().all()))

[PASS] N21-06: rows per family = {'gru_bestL': np.int64(72), 'lstm_L14': np.int64(72), 'lstm_L3': np.int64(72), 'lstm_L30': np.int64(72), 'lstm_L7': np.int64(72)} (expect 72 each); 360 family x task x horizon x season cells populated (min n_group=1); n all > 0 = True; bootstrap CI missing on big cells = 0; n_params present = True


In [8]:
# N21-07  headline: DL vs the D7 bar and the Phase-20 classical best, val_2022
val = met[(met.protocol == "P1") & (met.cell == "val_2022")
          & (met.season == "all")]
dl = val.pivot_table(index="model", columns="horizon", values="pr_auc").reindex(FAMILIES)
bar = pd.concat([
    p16[(p16.model == "persistence") & (p16.protocol == "P1")
        & (p16.cell == "val_2022") & (p16.season == "all")],
    p20[(p20.model == "xgboost") & (p20.protocol == "P1")
        & (p20.cell == "val_2022") & (p20.season == "all")],
]).pivot_table(index="model", columns="horizon", values="pr_auc")
allv = pd.concat([dl, bar])
print("val_2022 PR-AUC (season=all):")
print(allv.round(4).to_string())
best_dl = dl.max(axis=0)
delta_per = best_dl - bar.loc["persistence"]
delta_xgb = best_dl - bar.loc["xgboost"]
check("N21-07", f"best DL per horizon {best_dl.round(4).to_dict()}; vs D7 "
                f"persistence {delta_per.round(4).to_dict()}; vs xgboost "
                f"{delta_xgb.round(4).to_dict()}",
      True)

val_2022 PR-AUC (season=all):


horizon      label_24h  label_48h  label_72h
model                                       
lstm_L3         0.9421     0.9253     0.9129
lstm_L7         0.9309     0.9164     0.8871
lstm_L14        0.9311     0.9098     0.8823
lstm_L30        0.9388     0.9183     0.8936
gru_bestL       0.9359     0.9263     0.9086
persistence     0.9203     0.8954     0.8683
xgboost         0.9681     0.9549     0.9408
[PASS] N21-07: best DL per horizon {'label_24h': 0.9421, 'label_48h': 0.9263, 'label_72h': 0.9129}; vs D7 persistence {'label_24h': 0.0218, 'label_48h': 0.0309, 'label_72h': 0.0446}; vs xgboost {'label_24h': -0.0261, 'label_48h': -0.0286, 'label_72h': -0.0279}


In [9]:
# N21-08a  replay: each saved state_dict reproduces the recorded PR-AUC
from train_sequence import CHANNELS, SeqNet, fold_stats, prepare, std_rows

ix = pd.read_csv(PROC / "sequence_index.csv", parse_dates=["date"])
hold = ix.station_holdout.astype(bool).to_numpy()
split = ix.split.to_numpy()
tri = (split == "train") & ~hold
ev = (split == "val") & ~hold
labels = {h: ix[h].to_numpy() for h in HORIZONS}

bad = []
for f in FAMILIES:
    saved = torch.load(MODELS / f"phase21_{REG_ID[f]}.pt", weights_only=False)
    L = int(saved["L"])
    assert L == FAM_L[f], f
    X, M = prepare(L)
    kind = saved["kind"]
    p = saved["params"]
    for hor in HORIZONS:
        y = labels[hor]
        tr_idx = np.flatnonzero(tri & ~np.isnan(y))
        ev_idx = np.flatnonzero(ev & ~np.isnan(y))
        mu, sd = fold_stats(X, M, tr_idx)
        net = SeqNet(kind, p["hidden"], p["layers"], p["dropout"])
        net.load_state_dict(saved["state"][hor])
        net.eval()
        with torch.no_grad():
            pr = torch.sigmoid(net(torch.tensor(std_rows(X, M, ev_idx, mu, sd)))).numpy()
        ap = float(average_precision_score(y[ev_idx].astype(int), pr))
        rec = met[(met.model == f) & (met.cell == "val_2022")
                  & (met.horizon == hor) & (met.season == "all")].pr_auc.iloc[0]
        if abs(ap - rec) > 1e-9:
            bad.append((f, hor, ap, float(rec)))
check("N21-08a", f"all 5 x 3 saved state_dicts replay to the recorded PR-AUC "
                 f"exactly; mismatches={bad}", len(bad) == 0)

[PASS] N21-08a: all 5 x 3 saved state_dicts replay to the recorded PR-AUC exactly; mismatches=[]


In [10]:
# N21-08b  determinism: two identical fixed-epoch fits are bit-identical
from train_sequence import fit_model, seed_all

X, M = prepare(3)
y = labels["label_24h"]
tr_idx = np.flatnonzero(tri & ~np.isnan(y))[:4000]
ev_idx = np.flatnonzero(ev & ~np.isnan(y))[:2000]
mu, sd = fold_stats(X, M, tr_idx)
cfg = {"lr": 1e-3, "hidden": 32, "layers": 1, "dropout": 0.0,
       "batch_size": 256, "_weight": "off"}
r1 = fit_model("lstm", 3, cfg, X, M, mu, sd, tr_idx, y[tr_idx], ev_idx, y[ev_idx],
               fixed_epochs=2)
r2 = fit_model("lstm", 3, cfg, X, M, mu, sd, tr_idx, y[tr_idx], ev_idx, y[ev_idx],
               fixed_epochs=2)
same = all(torch.equal(r1["net"].state_dict()[k], r2["net"].state_dict()[k])
           for k in r1["net"].state_dict())
ap1 = float(average_precision_score(y[ev_idx].astype(int), r1["pred"]))
ap2 = float(average_precision_score(y[ev_idx].astype(int), r2["pred"]))
check("N21-08b", f"two seeded 2-epoch fits on 4,000 train rows: bit-identical "
                 f"parameters = {same}; val PR-AUC {ap1:.6f} == {ap2:.6f}",
      same and abs(ap1 - ap2) < 1e-12)

[PASS] N21-08b: two seeded 2-epoch fits on 4,000 train rows: bit-identical parameters = True; val PR-AUC 0.933876 == 0.933876


In [11]:
# N21-09  L* recorded and gru_bestL tuned at exactly L*
lstar = json.loads((PROC / "phase21_lstar.json").read_text())["L_star"]
lstm_mean = {FAM_L[f]: float(search[search.family == f].mean_pr_auc.max())
             for f in FAMILIES if f.startswith("lstm_")}
argmax_L = max(lstm_mean, key=lambda k: lstm_mean[k])
gru_L = int(search[search.family == "gru_bestL"].feature_set.iloc[0].split("L=")[-1])
check("N21-09", f"L* = {lstar}, argmax over the four LSTM families = {argmax_L} "
                f"(mean PR-AUC {lstm_mean}); gru_bestL tuned at L={gru_L}",
      int(lstar) == argmax_L and gru_L == int(lstar))

[PASS] N21-09: L* = 3, argmax over the four LSTM families = 3 (mean PR-AUC {3: 0.926768695140089, 7: 0.911464365597825, 14: 0.9077468659423484, 30: 0.9168824694504176}); gru_bestL tuned at L=3


In [12]:
# N21-10  headline figure: val_2022 PR-AUC by family vs the two reference bars
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

order = FAMILIES + ["xgboost", "persistence"]
plot = allv.reindex(order)
fig, ax = plt.subplots(figsize=(10, 4.6))
x = np.arange(len(order))
w = 0.26
for i, h in enumerate(HORIZONS):
    ax.bar(x + (i - 1) * w, plot.loc[order, h].to_numpy(), w, label=h)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=25, ha="right")
ax.set_ylabel("PR-AUC (val_2022)")
ax.set_ylim(0.8, 1.0)
ax.grid(alpha=0.3, axis="y")
ax.legend()
ax.set_title("Phase 21 - LSTM/GRU vs D7 bar and Phase-20 xgboost (val_2022, season=all)")
fig.tight_layout()
(ROOT / "figures").mkdir(exist_ok=True)
fig.savefig(ROOT / "figures" / "phase21_val2022.png", dpi=130)
plt.close(fig)
check("N21-10", f"report {REPORT.name} present ({REPORT.stat().st_size:,} B); "
                f"figure written to figures/phase21_val2022.png",
      REPORT.exists() and (ROOT / "figures" / "phase21_val2022.png").exists())

[PASS] N21-10: report phase21_temporal_dl.md present (9,593 B); figure written to figures/phase21_val2022.png


In [13]:
# N21-11  run log records a clean finish for every driver call
txt = LOG.read_text(encoding="utf-8", errors="replace")
clean = [ln.strip() for ln in txt.splitlines() if "exit=0" in ln]
check("N21-11", f"{LOG.name} {LOG.stat().st_size:,} B; clean-finish markers "
                f"(driver 'TOTAL … exit=0' + shell echoes): {len(clean)}",
      len(clean) >= 2)

[PASS] N21-11: phase21_run.log 25,481 B; clean-finish markers (driver 'TOTAL … exit=0' + shell echoes): 13


In [14]:
# N21-12/N21-13  byte-stability + summary
check("N21-12", f"experimental_dataset.csv SHA-256 = {sha256(TBL)[:12]}... "
                f"(unchanged: {sha256(TBL) == TBL_SHA})",
      sha256(TBL) == TBL_SHA)
print("\n" + "=" * 78)
print("PHASE 21 / STAGE 36  LSTM & GRU deep temporal models verified")
print("=" * 78)
for r in RES:
    print(f"  {'PASS' if r['pass'] else 'FAIL'}  {r['id']}")
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} checks passed, "
      f"{time.time() - t0:.1f}s")
print("L* = 3 (shortest window wins); test untouched (frozen to Phase 33); "
      "next: Phase 22 TCN (stage 37)")
assert all(r["pass"] for r in RES)
print("ALL PHASE-21 CHECKS PASSED")

[PASS] N21-12: experimental_dataset.csv SHA-256 = a6748529083f... (unchanged: True)

PHASE 21 / STAGE 36  LSTM & GRU deep temporal models verified
  PASS  N21-01
  PASS  N21-02
  PASS  N21-03
  PASS  N21-04
  PASS  N21-05
  PASS  N21-06
  PASS  N21-07
  PASS  N21-08a
  PASS  N21-08b
  PASS  N21-09
  PASS  N21-10
  PASS  N21-11
  PASS  N21-12

13/13 checks passed, 32.6s
L* = 3 (shortest window wins); test untouched (frozen to Phase 33); next: Phase 22 TCN (stage 37)
ALL PHASE-21 CHECKS PASSED
